In [1]:
#imports
#GPU
import sys, os
print("Python:", sys.executable)
print("LD_LIBRARY_PATH:", repr(os.environ.get("LD_LIBRARY_PATH")))
print("CUDA_HOME:", repr(os.environ.get("CUDA_HOME")))
print("CUDA_PATH:", repr(os.environ.get("CUDA_PATH")))

import jax
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

import os
from pathlib import Path
fdata='/home/dburrows/DATA/'


hf_home = Path(f"{fdata}/GENE_PREDICT/models/alphagenome/hf_home")
hf_home.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_home)
# optional, but makes it explicit:
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")

from alphagenome_research.model import dna_model
import numpy as np
import pandas as pd
import os

import jax
import alphagenome
import alphagenome_research
from alphagenome.data import genome
from alphagenome.models import dna_client
import time
import pysam
from tqdm import tqdm
import itertools

Python: /home/dburrows/venvs/alphagenome_research/bin/python
LD_LIBRARY_PATH: ''
CUDA_HOME: ''
CUDA_PATH: ''
JAX: 0.9.2
Devices: [CudaDevice(id=0)]


I0000 00:00:1777566630.550889  275946 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777566630.551780  275946 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777566631.372938  275946 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777566631.373465  275946 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [63]:
#imports
#GPU

import sys, os
print("Python:", sys.executable)
print("LD_LIBRARY_PATH:", repr(os.environ.get("LD_LIBRARY_PATH")))
print("CUDA_HOME:", repr(os.environ.get("CUDA_HOME")))
print("CUDA_PATH:", repr(os.environ.get("CUDA_PATH")))

import jax
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

import os
from pathlib import Path
fdata='/home/dburrows/DATA/'


hf_home = Path(f"{fdata}/GENE_PREDICT/models/alphagenome/hf_home")
hf_home.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_home)
# optional, but makes it explicit:
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")

from alphagenome_research.model import dna_model
import numpy as np
import pandas as pd
import os

import jax
import alphagenome
import alphagenome_research
from alphagenome.data import genome
from alphagenome.models import dna_client
import time
import pysam
from tqdm import tqdm
import itertools


#find locations, regions
#=========================
loc_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/4_SCN1A_variants_borzoi.bed', sep='\t').iloc[:,0].reset_index()
scn1a_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/1_SCN1A_gene.bed', sep='\t', header=None)
b1_df = pd.read_csv(f'{fdata}/GENE_PREDICT/importance_score/b1_20bp_single-ism_AG.csv', index_col=0)
b2_df = pd.read_csv(f'{fdata}/GENE_PREDICT/importance_score/b2_20bp_single-ism_AG.csv', index_col=0)
a1_df = pd.read_csv(f'{fdata}/GENE_PREDICT/importance_score/a1_20bp_single-ism_AG.csv', index_col=0)

#Load genome seq
#=================
fasta = pysam.FastaFile(f"{fdata}/GENOME/annotations/hg38/assembly/ucsc/hg38.fa")

#load models
#=========================
from alphagenome_research.model import dna_model
model = dna_model.create_from_huggingface("all_folds")
fold0 = dna_model.create_from_huggingface("fold_0")
fold1 = dna_model.create_from_huggingface("fold_1")
fold2 = dna_model.create_from_huggingface("fold_2")
fold3 = dna_model.create_from_huggingface("fold_3")
model_l = [fold0, fold1, fold2, fold3]
print("Models loaded.")


# mean expr

Python: /home/dburrows/venvs/alphagenome_research/bin/python
LD_LIBRARY_PATH: ''
CUDA_HOME: ''
CUDA_PATH: ''
JAX: 0.9.2
Devices: [CudaDevice(id=0)]


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


...

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Models loaded.


In [ ]:

#===============
def mean_reg(model=None, seq=None, chro=None, interval=None, brain_terms=None):
    vals = []
    for region_name, term in brain_terms.items():
        pred = model.predict_sequence(
            sequence=seq,
            requested_outputs=[dna_model.OutputType.RNA_SEQ],
            ontology_terms=[term],
            interval=interval,
        )
        vals.append(np.mean(pred.rna_seq.values, axis=1))
    return np.mean(np.array(vals), axis=0)


#expression across models
def expr_map(all_model = None, model_l=None, seq=None, chro=None, interval=None, brain_terms=None):
    point_est = mean_reg(model=all_model, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    fold_l = list(range(len(model_l)))
    for v,mo in enumerate(model_l):
        fold_l[v] = mean_reg(model=mo, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    return(point_est, np.array(fold_l))

#mutate arbitrary sequence
def mutate_seq(seq, pos, base):
    base = base.upper()
    if base not in {"A", "C", "G", "T"}:
        raise ValueError("base must be one of A, C, G, T")
    if pos < 0 or pos >= len(seq):
        raise IndexError("pos out of range")
    if seq[pos].upper() == base:
        raise ValueError("base is already the same at this position")

    seq = list(seq)
    seq[pos] = base
    return "".join(seq)

# mask sites to mutate over
def mask(mode=None, neg_thr=-0.03, neu_thr=(-0.03,0.06), reg=None, curr_df=None):
    mins = curr_df.groupby('pos')['logFC'].min()
    maxs = curr_df.groupby('pos')['logFC'].max()
    if mode == 'neg':
        keep_pos = mins[mins < neg_thr].index
    elif mode == 'neu':
        
        keep_pos = mins[(mins > neu_thr[0]) & (maxs < neu_thr[1])].index
    else:
        print('mode must equal neg or neu')
        return None

    return curr_df[curr_df['pos'].isin(keep_pos)].set_index('pos')

#get sequence window
def process(loc_df=None, scn1a_df = None, fasta = None, win_len=1048576, reg=None):
    curr_df = loc_df[loc_df['level_3'] == reg].copy()
    
    chro = str(curr_df['level_0'].iloc[0]) 
    region_start = int(curr_df['level_1'].iloc[0])
    region_end = int(curr_df['level_2'].iloc[0])
    cntr = (region_start + region_end) // 2
    
    start = cntr - (win_len//2)
    end = cntr + (win_len//2)
    interval = genome.Interval(chro, start, end)
    
    #mutation end and start relative to window
    region_rel_start = (region_start - start)
    region_rel_end = (region_end - start)
    
    #promoter end and start (for gene expression)
    gene_start = scn1a_df[1].values[0]
    gene_end = scn1a_df[2].values[0]
    gene_rel_start = (gene_start - start)
    gene_rel_end = (gene_end-start)
    
    
    print(interval)
    print("Width:", interval.width)

    seq = fasta.fetch(chro, start, end)
    return(chro, region_start, 
           region_end, cntr, interval, 
           region_rel_start, region_rel_end,
           gene_start, gene_end, gene_rel_start,
           gene_rel_end, seq)


def _run(reg, mode, curr_df):
    #models
    model_l = [fold0, fold1, fold2, fold3]
    
    # Brain-associated ontology terms 
    brain_terms = {
        "brain": "UBERON:0000955",
        "frontal_cortex": "UBERON:0001870",
        # "caudate_nucleus": "UBERON:0001873",
        # "putamen": "UBERON:0001874",
        # "amygdala": "UBERON:0001876",
        # "nucleus_accumbens": "UBERON:0001882",
        # "hypothalamus": "UBERON:0001898",
        "hippocampus_ammons_horn": "UBERON:0001954",
        # "cerebellum": "UBERON:0002037",
        # "substantia_nigra": "UBERON:0002038",
        # "cerebellar_hemisphere": "UBERON:0002245",
        # "spinal_cord_c1": "UBERON:0006469",
        "dlpfc_ba9": "UBERON:0009834",
        "anterior_cingulate_ba24": "UBERON:0009835",
    }
    
    # Extract sequence
    #===================
    (chro, region_start, region_end, cntr, interval, 
    region_rel_start, region_rel_end,
    gene_start, gene_end, gene_rel_start,
    gene_rel_end, seq 
    )= process(loc_df=loc_df, scn1a_df = scn1a_df, fasta = fasta, win_len=1048576, reg = reg)
    
    #generate baseline
    #======================
    baseline,_ = expr_map(all_model=model, model_l = model_l, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    base_mean = np.mean(baseline[gene_rel_start:gene_rel_end])

    
    
    # #Run ISM
    # #=================
    #Define mask
    neg_thr = -0.03
    neu_thr = (-0.03,0.06)
    mask_df = mask(mode = mode, neg_thr = neg_thr, neu_thr = neu_thr,
                   reg=reg, curr_df = curr_df)
    mask_rel_pos = np.array(mask_df.index.unique())
    
    
    # allowed alts per position from mask_df itself
    alt_by_pos = []
    ref_by_pos = []
    for pos in mask_rel_pos:
        sub = mask_df.loc[pos]
        sub = sub.sort_values('alt')
        ref_by_pos.append(sub['ref'].iloc[0])
        alt_by_pos.append(sub['alt'].tolist())
    
    print('n positions:', len(mask_rel_pos))
    print('total combos:', np.prod([len(x) for x in alt_by_pos]))
    
    # Combinatorial ISM
    # =================
    rows = []
    
    for combo in tqdm(itertools.product(*alt_by_pos), total=int(np.prod([len(x) for x in alt_by_pos]))):
        
        # mutate all selected positions at once
        mut_seq = list(seq)
        for pos, alt in zip(mask_rel_pos, combo):
            mut_seq[pos] = alt
        mut_seq = ''.join(mut_seq)
        
        # predict
        point, fold_outs = expr_map(
            all_model=model,
            model_l=model_l,
            seq=mut_seq,
            chro=chro,
            interval=interval,
            brain_terms=brain_terms
        )
        
        mut_mean = np.mean(point[gene_rel_start:gene_rel_end])
        logFC = np.log2((mut_mean + 1e-8) / (base_mean + 1e-8))
        sd = np.mean(np.std(fold_outs[:,gene_rel_start:gene_rel_end],axis=0))
        
        rows.append({
            'reg': reg,
            'mode': mode,
            'positions': tuple(mask_rel_pos),
            'refs': ''.join(ref_by_pos),
            'alts': ''.join(combo),
            'n_mut': len(combo),
            'mut_mean': mut_mean,
            'base_mean': base_mean,
            'delta': mut_mean - base_mean,
            'logFC': logFC,
            'sd': sd
        })
    
    comb_df = pd.DataFrame(rows).sort_values('logFC')
    comb_df.to_csv(f'{fdata}/GENE_PREDICT/ISM/{reg}_{mode}_20bp_alphagenome.csv')



reg_l = ['B2', 'B1', 'A1']
df_l = [b2_df, b1_df, a1_df]
mode_l = ['neg', 'neu']

for g,reg in enumerate(reg_l):
    curr_df = df_l[g]
    for mode in mode_l: 
        _run(reg,mode, curr_df)
        print(f'Done {reg} for {mode}') 

# Find high low regions

In [3]:
#find locations, regions
#=========================
loc_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/4_SCN1A_variants_borzoi.bed', sep='\t').iloc[:,0].reset_index()
scn1a_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/1_SCN1A_gene.bed', sep='\t', header=None)

#Load genome seq
#=================
fasta = pysam.FastaFile(f"{fdata}/GENOME/annotations/hg38/assembly/ucsc/hg38.fa")
loc_df

,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,"track name=""4_SCN1A_variants_borzoi"" description=""SCN1A variants borzoi"" visibility=2 itemRgb=""On"" color=255,0,0"
0,chr2,166127866,166127887,B1,0,-,166127867,166127887,"255,0,0"
1,chr2,166127501,166127520,B2,0,-,166127502,166127520,"255,0,0"
2,chr2,166149021,166149042,A1,0,-,166149022,166149042,"255,0,0"


In [ ]:
#load models
#=========================
from alphagenome_research.model import dna_model
model = dna_model.create_from_huggingface("all_folds")
fold0 = dna_model.create_from_huggingface("fold_0")
fold1 = dna_model.create_from_huggingface("fold_1")
fold2 = dna_model.create_from_huggingface("fold_2")
fold3 = dna_model.create_from_huggingface("fold_3")
model_l = [fold0, fold1, fold2, fold3]
print("Models loaded.")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


In [128]:
# mean expr
#===============
def mean_reg(model=None, seq=None, chro=None, interval=None, brain_terms=None):
    vals = []
    for region_name, term in brain_terms.items():
        pred = model.predict_sequence(
            sequence=seq,
            requested_outputs=[dna_model.OutputType.RNA_SEQ],
            ontology_terms=[term],
            interval=interval,
        )
        vals.append(np.mean(pred.rna_seq.values, axis=1))
    return np.mean(np.array(vals), axis=0)

#expression across models
def expr_map(all_model = None, model_l=None, seq=None, chro=None, interval=None, brain_terms=None):
    point_est = mean_reg(model=all_model, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    # fold_l = list(range(len(model_l)))
    # for v,mo in enumerate(model_l):
    #     fold_l[v] = mean_reg(model=mo, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    return(point_est)#, np.array(fold_l))

#mutate arbitrary sequence
def mutate_seq(seq, pos, base):
    base = base.upper()
    if base not in {"A", "C", "G", "T"}:
        raise ValueError("base must be one of A, C, G, T")
    if pos < 0 or pos >= len(seq):
        raise IndexError("pos out of range")
    if seq[pos].upper() == base:
        raise ValueError("base is already the same at this position")

    seq = list(seq)
    seq[pos] = base
    return "".join(seq)

#Run single ISM for importance score
def single_ism(model=None, model_l=None, seq=None, chro=None, interval=None, brain_terms=None,
               region_rel_start=None, region_rel_end=None,
               gene_rel_start=None, gene_rel_end=None, base_gene=None):
    bases = ["A", "C", "G", "T"]
    rows = []

    for pos in tqdm(np.arange(region_rel_start, region_rel_end)):
        for alt in bases:
            ref = seq[pos].upper()
            if alt == ref:
                continue
            mut_seq = mutate_seq(seq, pos, alt)
            score = expr_map(all_model=model, model_l = model_l, seq=mut_seq, chro=chro, interval=interval, brain_terms=brain_terms)

            score_gene = np.mean(score[gene_rel_start:gene_rel_end])
            fc = np.log2(score_gene / base_gene)       

            rows.append([pos, ref, alt, fc])

    out = pd.DataFrame(rows, columns=["pos", "ref", "alt", "logFC"])
    return out

def process(loc_df=None, scn1a_df = None, fasta = None, win_len=1048576, reg=None):
    curr_df = loc_df[loc_df['level_3'] == reg].copy()
    
    chro = str(curr_df['level_0'].iloc[0]) 
    region_start = int(curr_df['level_1'].iloc[0])
    region_end = int(curr_df['level_2'].iloc[0])
    cntr = (region_start + region_end) // 2
    win_len = 1048576
    
    start = cntr - (win_len//2)
    end = cntr + (win_len//2)
    interval = genome.Interval(chro, start, end)
    
    #mutation end and start relative to window
    region_rel_start = (region_start - start)
    region_rel_end = (region_end - start)
    
    #promoter end and start (for gene expression)
    gene_start = scn1a_df[1].values[0]
    gene_end = scn1a_df[2].values[0]
    gene_rel_start = (gene_start - start)
    gene_rel_end = (gene_end-start)
    
    
    print(interval)
    print("Width:", interval.width)

    seq = fasta.fetch(chro, start, end)
    return(chro, region_start, 
           region_end, cntr, interval, 
           region_rel_start, region_rel_end,
           gene_start, gene_end, gene_rel_start,
           gene_rel_end, seq)

In [142]:
#models
model_l = [fold0, fold1, fold2, fold3]

# Brain-associated ontology terms 
brain_terms = {
    "brain": "UBERON:0000955",
    "frontal_cortex": "UBERON:0001870",
    # "caudate_nucleus": "UBERON:0001873",
    # "putamen": "UBERON:0001874",
    # "amygdala": "UBERON:0001876",
    # "nucleus_accumbens": "UBERON:0001882",
    # "hypothalamus": "UBERON:0001898",
    "hippocampus_ammons_horn": "UBERON:0001954",
    # "cerebellum": "UBERON:0002037",
    # "substantia_nigra": "UBERON:0002038",
    # "cerebellar_hemisphere": "UBERON:0002245",
    # "spinal_cord_c1": "UBERON:0006469",
    "dlpfc_ba9": "UBERON:0009834",
    "anterior_cingulate_ba24": "UBERON:0009835",
}

# Extract sequence
#===================
reg = 'A1'
(chro, region_start, region_end, cntr, interval, 
region_rel_start, region_rel_end,
gene_start, gene_end, gene_rel_start,
gene_rel_end, seq 
)= process(loc_df=loc_df, scn1a_df = scn1a_df, fasta = fasta, win_len=1048576, reg = reg)

#generate baseline
baseline = expr_map(all_model=model, model_l=model_l, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
base_mean = np.mean(baseline[gene_rel_start:gene_rel_end])

#Run ism
outs = single_ism(
    model=model,
    model_l=model_l,
    seq=seq,
    chro=chro,
    interval=interval,
    brain_terms=brain_terms,
    region_rel_start=region_rel_start,
    region_rel_end=region_rel_end,
    gene_rel_start = gene_rel_start,
    gene_rel_end = gene_rel_end, 
    base_gene = base_mean
)
outs.to_csv(f'{fdata}/GENE_PREDICT/importance_score/a1_20bp_single-ism_AG.csv')

chr2:165624743-166673319:.
Width: 1048576


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [05:15<00:00, 15.01s/it]


In [141]:
loc_df

,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,"track name=""4_SCN1A_variants_borzoi"" description=""SCN1A variants borzoi"" visibility=2 itemRgb=""On"" color=255,0,0"
0,chr2,166127866,166127887,B1,0,-,166127867,166127887,"255,0,0"
1,chr2,166127501,166127520,B2,0,-,166127502,166127520,"255,0,0"
2,chr2,166149021,166149042,A1,0,-,166149022,166149042,"255,0,0"


In [137]:
b1_outs.to_csv(f'{fdata}/GENE_PREDICT/importance_score/b1_20bp_single-ism_AG.csv')

# Run ISM on high-low

In [2]:
#find locations, regions
#=========================
loc_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/4_SCN1A_variants_borzoi.bed', sep='\t').iloc[:,0].reset_index()
scn1a_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/1_SCN1A_gene.bed', sep='\t', header=None)
b1_df = pd.read_csv(f'{fdata}/GENE_PREDICT/importance_score/b1_20bp_single-ism_AG.csv', index_col=0)
b2_df = pd.read_csv(f'{fdata}/GENE_PREDICT/importance_score/b2_20bp_single-ism_AG.csv', index_col=0)
a1_df = pd.read_csv(f'{fdata}/GENE_PREDICT/importance_score/a1_20bp_single-ism_AG.csv', index_col=0)

#Load genome seq
#=================
fasta = pysam.FastaFile(f"{fdata}/GENOME/annotations/hg38/assembly/ucsc/hg38.fa")

In [4]:
# mask sites to mutate over
def mask(mode=None, neg_thr=-0.03, neu_thr=(-0.03,0.05), reg=None, curr_df=None):
    mins = curr_df.groupby('pos')['logFC'].min()
    maxs = curr_df.groupby('pos')['logFC'].max()
    if mode == 'neg':
        keep_pos = mins[mins < neg_thr].index
    elif mode == 'neu':
        
        keep_pos = mins[(mins > neu_thr[0]) & (maxs < neu_thr[1])].index
    else:
        print('mode must equal neg or neu')
        return None

    return curr_df[curr_df['pos'].isin(keep_pos)].set_index('pos')

In [5]:
a1_df

,pos,ref,alt,logFC
0,524278,T,A,0.019264
1,524278,T,C,0.016283
2,524278,T,G,0.038465
3,524279,G,A,0.007279
4,524279,G,C,-0.035990
...,...,...,...,...
58,524297,A,G,-0.117522
59,524297,A,T,-0.085811
60,524298,T,A,-0.026829
61,524298,T,C,0.258786


In [ ]:
neg_thr = -0.03
neu_thr = (-0.03,0.06)

In [18]:
len(np.unique(mask(mode = 'neg', neg_thr=-0.06, neu_thr=(-0.03,0.06), reg='A1', curr_df=a1_df).index))

7

In [58]:
pos = 524280
sub = mask_df.loc[pos]
sub = sub.sort_values('alt')
ref_by_pos.append(sub['ref'].iloc[0])
alt_by_pos.append(sub['alt'].tolist())

,ref,alt,logFC
pos,,,
524280,T,A,-0.059991
524280,T,C,-0.001248
524280,T,G,-0.002789


In [49]:
pos = 2

In [55]:
mask_df = mask(mode='neg', neg_thr=-0.03, neu_thr=(-0.03,0.05), reg='B1', curr_df=b1_df)
mask_df

,ref,alt,logFC
pos,,,
524280,T,A,-0.059991
524280,T,C,-0.001248
524280,T,G,-0.002789
524282,T,A,-0.038769
524282,T,C,-0.025812
524282,T,G,-0.031269
524283,T,A,-0.034799
524283,T,C,-0.001426
524283,T,G,-0.038454


In [41]:
len(np.unique(mask(mode='neg', neg_thr=-0.05, neu_thr=(-0.03,0.05), reg='B2', curr_df=b2_df).index))

7

In [40]:
len(np.unique(mask(mode='neg', neg_thr=-0.05, neu_thr=(-0.03,0.05), reg='A1', curr_df=a1_df).index))

11

In [31]:
len(np.unique(mask(mode='neu', neg_thr=-0.03, neu_thr=(-0.03,0.06), reg='B1', curr_df=b1_df).index))

9

In [32]:
len(np.unique(mask(mode='neu', neg_thr=-0.03, neu_thr=(-0.03,0.06), reg='B2', curr_df=b2_df).index))

6

In [33]:
len(np.unique(mask(mode='neu', neg_thr=-0.03, neu_thr=(-0.03,0.06), reg='A1', curr_df=a1_df).index))

6

In [7]:
b1_df

,pos,ref,alt,logFC
0,524278,C,A,-0.023514
1,524278,C,G,0.019919
2,524278,C,T,-0.007219
3,524279,T,A,0.006254
4,524279,T,C,0.002747
...,...,...,...,...
58,524297,A,G,0.057673
59,524297,A,T,0.047332
60,524298,A,C,0.081576
61,524298,A,G,0.036573


In [8]:
b1_df.groupby('pos')['logFC'].min()

pos
524278   -0.023514
524279    0.002747
524280   -0.059991
524281   -0.023018
524282   -0.038769
524283   -0.038454
524284   -0.022867
524285   -0.053183
524286   -0.046390
524287   -0.002807
524288   -0.358664
524289    0.030628
524290   -0.015983
524291   -0.147669
524292    0.039447
524293    0.038699
524294    0.028332
524295   -0.019824
524296    0.035541
524297    0.047332
524298    0.036573
Name: logFC, dtype: float64

In [ ]:
reg_l = ['B2', 'B1', 'A1']
df_l = [b2_df, b1_df, a1_df]
mode_l = ['neg', 'neu']

for g,reg in enumerate(reg_l):
    curr_df = df_l[g]
    for mode in mode_l: 
        _run(reg,mode, curr_df)
        print(f'Done {reg} for {mode}') 

In [6]:
mask(mode='neg', neg_thr=-0.03, neu_thr=(-0.03,0.05), reg='B1', curr_df=b1_df)

,ref,alt,logFC
pos,,,
524280,T,A,-0.059991
524280,T,C,-0.001248
524280,T,G,-0.002789
524282,T,A,-0.038769
524282,T,C,-0.025812
524282,T,G,-0.031269
524283,T,A,-0.034799
524283,T,C,-0.001426
524283,T,G,-0.038454


In [3]:
#load models
#=========================
from alphagenome_research.model import dna_model
model = dna_model.create_from_huggingface("all_folds")
fold0 = dna_model.create_from_huggingface("fold_0")
fold1 = dna_model.create_from_huggingface("fold_1")
fold2 = dna_model.create_from_huggingface("fold_2")
fold3 = dna_model.create_from_huggingface("fold_3")
model_l = [fold0, fold1, fold2, fold3]
print("Models loaded.")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

...

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Models loaded.


In [102]:
# mean expr
#===============
def mean_reg(model=None, seq=None, chro=None, interval=None, brain_terms=None):
    vals = []
    for region_name, term in brain_terms.items():
        pred = model.predict_sequence(
            sequence=seq,
            requested_outputs=[dna_model.OutputType.RNA_SEQ],
            ontology_terms=[term],
            interval=interval,
        )
        vals.append(np.mean(pred.rna_seq.values, axis=1))
    return np.mean(np.array(vals), axis=0)


#expression across models
def expr_map(all_model = None, model_l=None, seq=None, chro=None, interval=None, brain_terms=None):
    point_est = mean_reg(model=all_model, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    fold_l = list(range(len(model_l)))
    for v,mo in enumerate(model_l):
        fold_l[v] = mean_reg(model=mo, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    return(point_est, np.array(fold_l))

#mutate arbitrary sequence
def mutate_seq(seq, pos, base):
    base = base.upper()
    if base not in {"A", "C", "G", "T"}:
        raise ValueError("base must be one of A, C, G, T")
    if pos < 0 or pos >= len(seq):
        raise IndexError("pos out of range")
    if seq[pos].upper() == base:
        raise ValueError("base is already the same at this position")

    seq = list(seq)
    seq[pos] = base
    return "".join(seq)

# mask sites to mutate over
def mask(mode=None, neg_thr=-0.03, neu_thr=(-0.03,0.05), reg=None, curr_df=None):
    mins = curr_df.groupby('pos')['logFC'].min()

    if mode == 'neg':
        keep_pos = mins[mins < neg_thr].index
    elif mode == 'neu':
        keep_pos = mins[(mins > neu_thr[0]) & (mins < neu_thr[1])].index
    else:
        print('mode must equal neg or neu')
        return None

    return curr_df[curr_df['pos'].isin(keep_pos)].set_index('pos')

#get sequence window
def process(loc_df=None, scn1a_df = None, fasta = None, win_len=1048576, reg=None):
    curr_df = loc_df[loc_df['level_3'] == reg].copy()
    
    chro = str(curr_df['level_0'].iloc[0]) 
    region_start = int(curr_df['level_1'].iloc[0])
    region_end = int(curr_df['level_2'].iloc[0])
    cntr = (region_start + region_end) // 2
    win_len = 1048576
    
    start = cntr - (win_len//2)
    end = cntr + (win_len//2)
    interval = genome.Interval(chro, start, end)
    
    #mutation end and start relative to window
    region_rel_start = (region_start - start)
    region_rel_end = (region_end - start)
    
    #promoter end and start (for gene expression)
    gene_start = scn1a_df[1].values[0]
    gene_end = scn1a_df[2].values[0]
    gene_rel_start = (gene_start - start)
    gene_rel_end = (gene_end-start)
    
    
    print(interval)
    print("Width:", interval.width)

    seq = fasta.fetch(chro, start, end)
    return(chro, region_start, 
           region_end, cntr, interval, 
           region_rel_start, region_rel_end,
           gene_start, gene_end, gene_rel_start,
           gene_rel_end, seq)


def _run(reg, mode, curr_df):
    #models
    model_l = [fold0, fold1, fold2, fold3]
    
    # Brain-associated ontology terms 
    brain_terms = {
        "brain": "UBERON:0000955",
        "frontal_cortex": "UBERON:0001870",
        # "caudate_nucleus": "UBERON:0001873",
        # "putamen": "UBERON:0001874",
        # "amygdala": "UBERON:0001876",
        # "nucleus_accumbens": "UBERON:0001882",
        # "hypothalamus": "UBERON:0001898",
        "hippocampus_ammons_horn": "UBERON:0001954",
        # "cerebellum": "UBERON:0002037",
        # "substantia_nigra": "UBERON:0002038",
        # "cerebellar_hemisphere": "UBERON:0002245",
        # "spinal_cord_c1": "UBERON:0006469",
        "dlpfc_ba9": "UBERON:0009834",
        "anterior_cingulate_ba24": "UBERON:0009835",
    }
    
    # Extract sequence
    #===================
    (chro, region_start, region_end, cntr, interval, 
    region_rel_start, region_rel_end,
    gene_start, gene_end, gene_rel_start,
    gene_rel_end, seq 
    )= process(loc_df=loc_df, scn1a_df = scn1a_df, fasta = fasta, win_len=1048576, reg = reg)
    
    #generate baseline
    #======================
    baseline,_ = expr_map(all_model=model, model_l = model_l, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    base_mean = np.mean(baseline[gene_rel_start:gene_rel_end])
    
    # #Run ISM
    # #=================
    #Define mask
    neg_thr = -0.03
    neu_thr = (-0.03,0.05)
    mask_df = mask(mode = mode, neg_thr = neg_thr, neu_thr = neu_thr,
                   reg=reg, curr_df = curr_df)
    mask_rel_pos = np.array(mask_df.index.unique())
    
    
    # allowed alts per position from mask_df itself
    alt_by_pos = []
    ref_by_pos = []
    for pos in mask_rel_pos:
        sub = mask_df.loc[pos]
        sub = sub.sort_values('alt')
        ref_by_pos.append(sub['ref'].iloc[0])
        alt_by_pos.append(sub['alt'].tolist())
    
    print('n positions:', len(mask_rel_pos))
    print('total combos:', np.prod([len(x) for x in alt_by_pos]))
    
    # Combinatorial ISM
    # =================
    rows = []
    
    for combo in tqdm(itertools.product(*alt_by_pos), total=int(np.prod([len(x) for x in alt_by_pos]))):
        
        # mutate all selected positions at once
        mut_seq = list(seq)
        for pos, alt in zip(mask_rel_pos, combo):
            mut_seq[pos] = alt
        mut_seq = ''.join(mut_seq)
        
        # predict
        point, fold_outs = expr_map(
            all_model=model,
            model_l=model_l,
            seq=mut_seq,
            chro=chro,
            interval=interval,
            brain_terms=brain_terms
        )
        
        mut_mean = np.mean(point[gene_rel_start:gene_rel_end])
        logFC = np.log2((mut_mean + 1e-8) / (base_mean + 1e-8))
        sd = np.mean(np.std(fold_outs[:,gene_rel_start:gene_rel_end],axis=0))
        
        rows.append({
            'reg': reg,
            'mode': mode,
            'positions': tuple(mask_rel_pos),
            'refs': ''.join(ref_by_pos),
            'alts': ''.join(combo),
            'n_mut': len(combo),
            'mut_mean': mut_mean,
            'base_mean': base_mean,
            'delta': mut_mean - base_mean,
            'logFC': logFC,
            'sd': sd
        })
    
    comb_df = pd.DataFrame(rows).sort_values('logFC')
    comb_df.to_csv(f'{fdata}/GENE_PREDICT/ISM/{reg}_{mode}_20bp_alphagenome.csv')


In [ ]:
reg_l = ['B2', 'B1', 'A1']
df_l = [b2_df, b1_df, a1_df]
mode_l = ['neg', 'neu']

for g,reg in enumerate(reg_l):
    curr_df = df_l[g]
    for mode in mode_l: 
        _run(reg,mode, curr_df)
        print(f'Done {reg} for {mode}') 

# Second, neutral only, matched length run, 10bp run

In [141]:
#imports
#GPU

import sys, os
print("Python:", sys.executable)
print("LD_LIBRARY_PATH:", repr(os.environ.get("LD_LIBRARY_PATH")))
print("CUDA_HOME:", repr(os.environ.get("CUDA_HOME")))
print("CUDA_PATH:", repr(os.environ.get("CUDA_PATH")))

import jax
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

import os
from pathlib import Path
fdata='/home/dburrows/DATA/'


hf_home = Path(f"{fdata}/GENE_PREDICT/models/alphagenome/hf_home")
hf_home.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_home)
# optional, but makes it explicit:
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")

from alphagenome_research.model import dna_model
import numpy as np
import pandas as pd
import os

import jax
import alphagenome
import alphagenome_research
from alphagenome.data import genome
from alphagenome.models import dna_client
import time
import pysam
from tqdm import tqdm
import itertools


#find locations, regions
#=========================
loc_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/4_SCN1A_variants_borzoi.bed', sep='\t').iloc[:,0].reset_index()
scn1a_df=pd.read_csv(f'{fdata}/GENOME/scn1a/scn1a_aaron/1_SCN1A_gene.bed', sep='\t', header=None)

#Load genome seq
#=================
fasta = pysam.FastaFile(f"{fdata}/GENOME/annotations/hg38/assembly/ucsc/hg38.fa")

#load models
#=========================
from alphagenome_research.model import dna_model
model = dna_model.create_from_huggingface("all_folds")
print("Models loaded.")


# mean expr
#===============
def mean_reg(model=None, seq=None, chro=None, interval=None, brain_terms=None):
    vals = []
    for region_name, term in brain_terms.items():
        pred = model.predict_sequence(
            sequence=seq,
            requested_outputs=[dna_model.OutputType.RNA_SEQ],
            ontology_terms=[term],
            interval=interval,
        )
        vals.append(np.mean(pred.rna_seq.values, axis=1))
    return np.mean(np.array(vals), axis=0)

#expression across models
def expr_map(all_model = None, model_l=None, seq=None, chro=None, interval=None, brain_terms=None):
    point_est = mean_reg(model=all_model, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    # fold_l = list(range(len(model_l)))
    # for v,mo in enumerate(model_l):
    #     fold_l[v] = mean_reg(model=mo, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    return(point_est)#, np.array(fold_l))

#mutate arbitrary sequence
def mutate_seq(seq, pos, base):
    base = base.upper()
    if base not in {"A", "C", "G", "T"}:
        raise ValueError("base must be one of A, C, G, T")
    if pos < 0 or pos >= len(seq):
        raise IndexError("pos out of range")
    if seq[pos].upper() == base:
        raise ValueError("base is already the same at this position")

    seq = list(seq)
    seq[pos] = base
    return "".join(seq)

#get sequence window
def process(loc_df=None, scn1a_df = None, fasta = None, win_len=1048576, reg=None):
    curr_df = loc_df[loc_df['level_3'] == reg].copy()
    
    chro = str(curr_df['level_0'].iloc[0]) 
    region_start = int(curr_df['level_1'].iloc[0])
    region_end = int(curr_df['level_2'].iloc[0])
    cntr = (region_start + region_end) // 2
    
    start = cntr - (win_len//2)
    end = cntr + (win_len//2)
    interval = genome.Interval(chro, start, end)
    
    #mutation end and start relative to window
    region_rel_start = (region_start - start)
    region_rel_end = (region_end - start)
    
    #promoter end and start (for gene expression)
    gene_start = scn1a_df[1].values[0]
    gene_end = scn1a_df[2].values[0]
    gene_rel_start = (gene_start - start)
    gene_rel_end = (gene_end-start)
    
    
    print(interval)
    print("Width:", interval.width)

    seq = fasta.fetch(chro, start, end)
    return(chro, region_start, 
           region_end, cntr, interval, 
           region_rel_start, region_rel_end,
           gene_start, gene_end, gene_rel_start,
           gene_rel_end, seq)


def prepare_region(loc_df=None, scn1a_df=None, fasta=None, win_len=1048576, reg=None):
    brain_terms = {
        "brain": "UBERON:0000955",
        "frontal_cortex": "UBERON:0001870",
        "hippocampus_ammons_horn": "UBERON:0001954",
        "dlpfc_ba9": "UBERON:0009834",
        "anterior_cingulate_ba24": "UBERON:0009835",
    }

    (
        chro, region_start, region_end, cntr, interval,
        region_rel_start, region_rel_end,
        gene_start, gene_end, gene_rel_start,
        gene_rel_end, seq
    ) = process(
        loc_df=loc_df,
        scn1a_df=scn1a_df,
        fasta=fasta,
        win_len=win_len,
        reg=reg,
    )

    baseline = expr_map(
        all_model=model,
        seq=seq,
        chro=chro,
        interval=interval,
        brain_terms=brain_terms,
    )

    base_mean = np.mean(baseline[gene_rel_start:gene_rel_end])

    return {
        "reg": reg,
        "brain_terms": brain_terms,

        "chro": chro,
        "region_start": int(region_start),
        "region_end": int(region_end),
        "cntr": int(cntr),
        "interval": interval,

        "region_rel_start": int(region_rel_start),
        "region_rel_end": int(region_rel_end),

        "gene_start": int(gene_start),
        "gene_end": int(gene_end),
        "gene_rel_start": int(gene_rel_start),
        "gene_rel_end": int(gene_rel_end),

        "seq": seq,
        "baseline": baseline,
        "base_mean": float(base_mean),
    }


def run_mutation(region_info, N=10):
    reg = region_info["reg"]
    brain_terms = region_info["brain_terms"]

    chro = region_info["chro"]
    interval = region_info["interval"]

    region_start = region_info["region_start"]
    region_end = region_info["region_end"]
    region_rel_start = region_info["region_rel_start"]
    region_rel_end = region_info["region_rel_end"]

    gene_rel_start = region_info["gene_rel_start"]
    gene_rel_end = region_info["gene_rel_end"]

    seq = region_info["seq"]
    base_mean = region_info["base_mean"]

    four = np.asarray(["A", "C", "G", "T"])

    start_pos = int(region_rel_start)
    end_pos = int(region_rel_end) - 1  # region_rel_end is exclusive

    if end_pos <= start_pos:
        raise ValueError("Region is too short to mutate start and end.")

    interior_positions = np.arange(start_pos + 1, end_pos)
    n_random = N - 2

    if n_random > len(interior_positions):
        raise ValueError(
            f"Need {n_random} random interior positions, "
            f"but only {len(interior_positions)} available."
        )

    random_positions = np.random.choice(
        interior_positions,
        size=n_random,
        replace=False,
    )

    # Force start + end, plus random interior positions
    its = np.concatenate([[start_pos, end_pos], random_positions]).astype(int)
    np.random.shuffle(its)

    assert len(its) == N
    assert len(its) == len(np.unique(its)), "its contains duplicate positions"
    assert start_pos in its
    assert end_pos in its

    mutseq = seq

    refs = []
    alts = []
    positions_abs_window = []
    positions_rel_region = []

    for it in its:
        it = int(it)

        curr = mutseq[it].upper()

        if curr not in {"A", "C", "G", "T"}:
            raise ValueError(f"Non-ACGT base at position {it}: {curr}")

        not_curr = four[four != curr]
        base = str(np.random.choice(not_curr))

        refs.append(curr)
        alts.append(base)
        positions_abs_window.append(it)
        positions_rel_region.append(it - region_rel_start)

        mutseq = mutate_seq(mutseq, it, base)

    # Sanity checks
    all_changed = np.array([
        i for i, (a, b) in enumerate(zip(seq.upper(), mutseq.upper()))
        if a != b
    ])

    assert len(all_changed) == N, f"Expected exactly {N} total changes, found {len(all_changed)}"
    assert set(map(int, all_changed)) == set(map(int, its)), "Changed positions do not match requested positions"
    assert seq[start_pos].upper() != mutseq[start_pos].upper(), "Start position did not mutate"
    assert seq[end_pos].upper() != mutseq[end_pos].upper(), "End position did not mutate"

    # Mutant prediction only
    point = expr_map(
        all_model=model,
        seq=mutseq,
        chro=chro,
        interval=interval,
        brain_terms=brain_terms,
    )

    mut_mean = np.mean(point[gene_rel_start:gene_rel_end])

    FC = (mut_mean + 1e-8) / (base_mean + 1e-8)
    logFC = np.log2(FC)

    return {
        "reg": reg,
        "n_mut": N,

        # mutation info
        "positions_window": tuple(map(int, positions_abs_window)),
        "positions_region": tuple(map(int, positions_rel_region)),
        "refs": "".join(refs),
        "alts": "".join(alts),

        # region info
        "chro": chro,
        "region_start": int(region_start),
        "region_end": int(region_end),
        "region_rel_start": int(region_rel_start),
        "region_rel_end": int(region_rel_end),
        "start_forced_pos": int(start_pos),
        "end_forced_pos": int(end_pos),

        # prediction info
        "base_mean": float(base_mean),
        "mut_mean": float(mut_mean),
        "delta": float(mut_mean - base_mean),
        "FC": float(FC),
        "logFC": float(logFC),
    }

Python: /home/dburrows/venvs/alphagenome_research/bin/python
LD_LIBRARY_PATH: ''
CUDA_HOME: ''
CUDA_PATH: ''
JAX: 0.9.2
Devices: [CudaDevice(id=0)]


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/home/dburrows/venvs/alphagenome_research/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


Models loaded.


In [140]:
loc_df

,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,"track name=""4_SCN1A_variants_borzoi"" description=""SCN1A variants borzoi"" visibility=2 itemRgb=""On"" color=255,0,0"
0,chr2,166127866,166127887,B1,0,-,166127867,166127887,"255,0,0"
1,chr2,166127501,166127520,B2,0,-,166127502,166127520,"255,0,0"
2,chr2,166149021,166149042,A1,0,-,166149022,166149042,"255,0,0"


In [ ]:
# ============================================================
# Search settings
# ============================================================

reg_l = ["B2", "B1", "A1"]

N = 10

fc_low = 0.95
fc_high = 1.05

target_keep_per_reg = 20
max_attempts_per_reg = 500

mode = f"{N}mut_neutral_fc_{fc_low}_{fc_high}"

out_dir = f"{fdata}/GENE_PREDICT/ISM"
os.makedirs(out_dir, exist_ok=True)

out_path = f"{out_dir}/scn1a_{mode}_accepted.csv"

rows = []

# ============================================================
# Run search
# ============================================================

for reg in reg_l:
    kept = 0
    attempted = 0

    print(f"\n=== Preparing {reg} baseline ===")

    region_info = prepare_region(
        loc_df=loc_df,
        scn1a_df=scn1a_df,
        fasta=fasta,
        win_len=1048576,
        reg=reg,
    )

    print(f"{reg} base_mean:", region_info["base_mean"])
    print(f"\n=== Starting {reg} mutation search ===")

    pbar = tqdm(total=target_keep_per_reg, desc=f"{reg} kept")

    while kept < target_keep_per_reg and attempted < max_attempts_per_reg:
        attempted += 1

        out = run_mutation(
            region_info=region_info,
            N=N,
        )

        fc = out["FC"]

        if fc_low <= fc <= fc_high:
            kept += 1

            out["mode"] = mode
            out["attempt"] = attempted
            out["kept_idx"] = kept
            out["accepted"] = True

            rows.append(out)

            pbar.update(1)
            pbar.set_postfix({"attempts": attempted, "FC": round(fc, 4)})

            # Save after every accepted row
            keep_df = pd.DataFrame(rows)
            keep_df.to_csv(out_path, index=False)

        else:
            if attempted % 10 == 0:
                print(
                    f"Rejected {reg} attempt {attempted}: "
                    f"FC={fc:.4f}, logFC={out['logFC']:.4f}, kept={kept}"
                )

    pbar.close()

    print(
        f"Done {reg}: kept {kept}/{attempted} attempts "
        f"within FC range [{fc_low}, {fc_high}]"
    )

# ============================================================
# Final save
# ============================================================

keep_df = pd.DataFrame(rows)
keep_df.to_csv(out_path, index=False)

print(f"\nSaved accepted constructs to: {out_path}")
print(keep_df.head())
print("Total accepted:", len(keep_df))

if len(keep_df) > 0:
    print(keep_df.groupby("reg").size())
else:
    print("No accepted constructs.")

In [10]:
loc_df

,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,"track name=""4_SCN1A_variants_borzoi"" description=""SCN1A variants borzoi"" visibility=2 itemRgb=""On"" color=255,0,0"
0,chr2,166127866,166127887,B1,0,-,166127867,166127887,"255,0,0"
1,chr2,166127501,166127520,B2,0,-,166127502,166127520,"255,0,0"
2,chr2,166149021,166149042,A1,0,-,166149022,166149042,"255,0,0"


In [16]:
reg = 'A1'

In [136]:
logFC

np.float32(-0.1267136)

In [ ]:
# allowed alts per position from mask_df itself
alt_by_pos = []
ref_by_pos = []
for pos in mask_rel_pos:
    sub = mask_df.loc[pos]
    sub = sub.sort_values('alt')
    ref_by_pos.append(sub['ref'].iloc[0])
    alt_by_pos.append(sub['alt'].tolist())

print('n positions:', len(mask_rel_pos))
print('total combos:', np.prod([len(x) for x in alt_by_pos]))

# Combinatorial ISM
# =================
rows = []

for combo in tqdm(itertools.product(*alt_by_pos), total=int(np.prod([len(x) for x in alt_by_pos]))):
    
    # mutate all selected positions at once
    mut_seq = list(seq)
    for pos, alt in zip(mask_rel_pos, combo):
        mut_seq[pos] = alt
    mut_seq = ''.join(mut_seq)
    
    # predict
    point = expr_map(
        all_model=model,
        #model_l=model_l,
        seq=mut_seq,
        chro=chro,
        interval=interval,
        brain_terms=brain_terms
    )
    
    mut_mean = np.mean(point[gene_rel_start:gene_rel_end])
    logFC = np.log2((mut_mean + 1e-8) / (base_mean + 1e-8))
    #sd = np.mean(np.std(fold_outs[:,gene_rel_start:gene_rel_end],axis=0))
    

In [ ]:
# # mask sites to mutate over
# def mask(mode=None, neg_thr=-0.03, neu_thr=(-0.03,0.06), reg=None, curr_df=None):
#     mins = curr_df.groupby('pos')['logFC'].min()
#     maxs = curr_df.groupby('pos')['logFC'].max()
#     if mode == 'neg':
#         keep_pos = mins[mins < neg_thr].index
#     elif mode == 'neu':
        
#         keep_pos = mins[(mins > neu_thr[0]) & (maxs < neu_thr[1])].index
#     else:
#         print('mode must equal neg or neu')
#         return None

#     return curr_df[curr_df['pos'].isin(keep_pos)].set_index('pos')



def _run(reg, mode, curr_df):
    #models
    #model_l = [fold0, fold1, fold2, fold3]
    
    # Brain-associated ontology terms 
    brain_terms = {
        "brain": "UBERON:0000955",
        "frontal_cortex": "UBERON:0001870",
        # "caudate_nucleus": "UBERON:0001873",
        # "putamen": "UBERON:0001874",
        # "amygdala": "UBERON:0001876",
        # "nucleus_accumbens": "UBERON:0001882",
        # "hypothalamus": "UBERON:0001898",
        "hippocampus_ammons_horn": "UBERON:0001954",
        # "cerebellum": "UBERON:0002037",
        # "substantia_nigra": "UBERON:0002038",
        # "cerebellar_hemisphere": "UBERON:0002245",
        # "spinal_cord_c1": "UBERON:0006469",
        "dlpfc_ba9": "UBERON:0009834",
        "anterior_cingulate_ba24": "UBERON:0009835",
    }
    
    # Extract sequence
    #===================
    (chro, region_start, region_end, cntr, interval, 
    region_rel_start, region_rel_end,
    gene_start, gene_end, gene_rel_start,
    gene_rel_end, seq 
    )= process(loc_df=loc_df, scn1a_df = scn1a_df, fasta = fasta, win_len=1048576, reg = reg)
    
    #generate baseline
    #======================
    baseline = expr_map(all_model=model, seq=seq, chro=chro, interval=interval, brain_terms=brain_terms)
    base_mean = np.mean(baseline[gene_rel_start:gene_rel_end])
    
    # #Run ISM
    # #=================
    #Define mask
    neg_thr = -0.03
    neu_thr = (-0.03,0.06)
    mask_df = mask(mode = mode, neg_thr = neg_thr, neu_thr = neu_thr,
                   reg=reg, curr_df = curr_df)
    mask_rel_pos = np.array(mask_df.index.unique())
    
    
    # allowed alts per position from mask_df itself
    alt_by_pos = []
    ref_by_pos = []
    for pos in mask_rel_pos:
        sub = mask_df.loc[pos]
        sub = sub.sort_values('alt')
        ref_by_pos.append(sub['ref'].iloc[0])
        alt_by_pos.append(sub['alt'].tolist())
    
    print('n positions:', len(mask_rel_pos))
    print('total combos:', np.prod([len(x) for x in alt_by_pos]))
    
    # Combinatorial ISM
    # =================
    rows = []
    
    for combo in tqdm(itertools.product(*alt_by_pos), total=int(np.prod([len(x) for x in alt_by_pos]))):
        
        # mutate all selected positions at once
        mut_seq = list(seq)
        for pos, alt in zip(mask_rel_pos, combo):
            mut_seq[pos] = alt
        mut_seq = ''.join(mut_seq)
        
        # predict
        point = expr_map(
            all_model=model,
            #model_l=model_l,
            seq=mut_seq,
            chro=chro,
            interval=interval,
            brain_terms=brain_terms
        )
        
        mut_mean = np.mean(point[gene_rel_start:gene_rel_end])
        logFC = np.log2((mut_mean + 1e-8) / (base_mean + 1e-8))
        #sd = np.mean(np.std(fold_outs[:,gene_rel_start:gene_rel_end],axis=0))
        
        rows.append({
            'reg': reg,
            'mode': mode,
            'positions': tuple(mask_rel_pos),
            'refs': ''.join(ref_by_pos),
            'alts': ''.join(combo),
            'n_mut': len(combo),
            'mut_mean': mut_mean,
            'base_mean': base_mean,
            'delta': mut_mean - base_mean,
            'logFC': logFC,
            #'sd': sd
        })
    
    comb_df = pd.DataFrame(rows).sort_values('logFC')
    comb_df.to_csv(f'{fdata}/GENE_PREDICT/ISM/{reg}_{mode}_20bp_alphagenome.csv')

reg_l = ['B2', 'B1', 'A1']
df_l = [b2_df, b1_df, a1_df]
mode_l = ['neg', 'neu']

for g,reg in enumerate(reg_l):
    curr_df = df_l[g]
    for mode in mode_l: 
        _run(reg,mode, curr_df)
        print(f'Done {reg} for {mode}') 